<p style="background-color:#f8f9fa; 
          padding:18px; 
          color:#111;
          font-size:16px;
          border-width:2px; 
          border-color:#e0e0e0; 
          border-style:solid;
          border-radius:6px">
⛳ <strong>Surgical ONNX: Precision Parameter Reduction</strong><br><br>
After reviewing several ONNX networks submitted to NeuroGolf, I kept finding the same hidden waste — parameters that inflate the cost but contribute nothing to the computation.<br><br>
This notebook applies <strong>three automated fixes</strong>, each one zero-risk:<br><br>
✂️ <strong>Prune</strong> — remove initializers that no node reads<br>
🔗 <strong>Dedup</strong> — merge identical tensors stored under different names<br>
📡 <strong>Compress</strong> — replace uniform arrays with scalars via broadcasting<br><br>🎯 Same outputs. Fewer params. Higher score.
</p>

# ⚙️ Setup

## 🖥️ Env

In [ ]:
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null

$$ $$

## 🧬 Load Best Public Submission

In [ ]:
!rm -r /kaggle/working/submission 2>/dev/null
!cp /kaggle/input/notebooks/rajathrpai/neurogolf-6347-80/submission.zip /kaggle/working/submission.zip
!unzip -q /kaggle/working/submission.zip -d /kaggle/working/submission
!rm /kaggle/working/submission.zip 
!ls -la

In [ ]:
SRC = "/kaggle/working/submission/overrides"

$$ $$

## 📦 Imports

In [ ]:
import sys
sys.path.append("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
from neurogolf_utils import *
show_legend()

In [ ]:
import math, glob, os
import copy
from tqdm.auto import tqdm
from collections import defaultdict, Counter


import pandas as pd
import numpy as np


import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime

$$ $$

## ƒ Utils

In [ ]:
def score_task(task_num, network):
    """Returns (points, memory, params) or None if failed."""
    filename = f"task{task_num:03d}.onnx"
    onnx.save(network, filename)
    if not check_network(filename):
        return None

    try:
        sanitized = onnx.load(filename)
        for node in sanitized.graph.node:
            node.name = node.output[0]
            if "kernel_time" in node.name:
                return None
        options = onnxruntime.SessionOptions()
        options.enable_profiling = True
        options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
        options.profile_file_prefix = f"{task_num:03}"
        session = onnxruntime.InferenceSession(sanitized.SerializeToString(), options)
    except Exception:
        return None

    examples = load_examples(task_num)
    agi_r, agi_w, _ = verify_subset(session, examples["train"] + examples["test"])
    gen_r, gen_w, _ = verify_subset(session, examples["arc-gen"])

    memory, params = score_network(sanitized, session.end_profiling())
    if memory is None or params is None or memory < 0 or params < 0:
        return None

    if agi_w + gen_w > 0:
        return None  # incorrect

    points = max(1.0, 25.0 - math.log(max(1.0, memory + params)))
    return points, memory, params

$$ $$

# 🧹 Optimizations

## ✂️ Prune Unused Initializers

<p style="background-color:#e6f7ff; 
          padding:15px; 
          color:#111;
          font-size:16px;
          border-width:3px; 
          border-color:#d0eefc; 
          border-style:solid;
          border-radius:6px">
✂️ <strong>Prune Unused Initializers</strong><br><br>
ONNX networks sometimes contain <code>initializers</code> (constant tensors) that no node actually reads. The NeuroGolf scorer counts <strong>every</strong> initializer element as a parameter — whether used or not:<br><br>
<code>cost = total_parameters + total_memory_in_bytes</code><br>
<code>score = max(1, 25 - ln(cost))</code><br><br>
An unused <code>[30]</code> tensor silently adds <strong>30</strong> to the cost and lowers the score for <em>zero benefit</em>.<br><br>
<strong>What this notebook does:</strong><br>
1️⃣ Loads each <code>.onnx</code> file from the submission<br>
2️⃣ Finds initializers <em>not referenced</em> by any node<br>
3️⃣ Removes them and re-saves<br>
4️⃣ Re-zips for submission<br><br>
No network behavior changes. Same outputs, fewer params, <strong>higher score</strong>. Free points. 🎯
</p>


<p style="background-color:#fff3e0; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #ff9800;
          border-radius:4px">
🔍 <strong>Step 1: How bad is it?</strong><br>
Let's scan all 400 models and find out how many are carrying dead weight.
</p>

In [ ]:
worst_tasks = []

for path in tqdm(sorted(glob.glob(f"{SRC}/task*.onnx"))):
    task_num = int(os.path.basename(path).replace("task","").replace(".onnx",""))
    model = onnx.load(path)
    
    used = set()
    for node in model.graph.node:
        used.update(inp for inp in node.input if inp)
    
    wasted = []
    for init in model.graph.initializer:
        if init.name not in used:
            dims = list(init.dims) if init.dims else [1]
            wasted.append((init.name, dims, int(np.prod(dims))))
    
    if wasted:
        total_wasted = sum(w[2] for w in wasted)
        total_params = sum(int(np.prod(i.dims)) if i.dims else 1 
                          for i in model.graph.initializer)
        worst_tasks.append((task_num, total_wasted, total_params, wasted))

# Sort by most wasted params
worst_tasks.sort(key=lambda x: -x[1])

print(f"📊 {len(worst_tasks)} tasks have unused initializers\n")
print(f"{'Task':>6} {'Wasted':>8} {'Total':>8} {'% Waste':>8}   Unused tensors")
print(f"{'─'*6} {'─'*8} {'─'*8} {'─'*8}   {'─'*30}")
for task_num, wasted, total, details in worst_tasks[:15]:
    pct = 100 * wasted / total if total else 0
    names = ", ".join(f"'{d[0]}'" for d in details[:3])
    if len(details) > 3:
        names += f" +{len(details)-3} more"
    print(f"  {task_num:03d}    {wasted:>6}   {total:>6}   {pct:>5.1f}%   {names}")

<p style="background-color:#e8f5e9; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #4caf50;
          border-radius:4px">
🔬 <strong>Step 2: Fix one, prove it works</strong><br>
Pick the worst offender. Remove the dead initializers. Verify the score improves and outputs don't change.
</p>

In [ ]:
# 🔬 Let's look at Task XXX in detail
if worst_tasks:
    t = worst_tasks[0][0]
    network = onnx.load(f"{SRC}/task{t:03d}.onnx")
    
    score_before, mem_before, params_before = score_task(t, network)
    print(f"Task {t:03d} BEFORE cleanup:")
    print(f"  Parameters: {params_before}")
    print(f"  Memory:     {mem_before} bytes")
    print(f"  Score:      {score_before:.4f}")
    print(f"  Unused initializers:")
    
    used = set()
    for node in network.graph.node:
        used.update(inp for inp in node.input if inp)
    
    for init in network.graph.initializer:
        if init.name not in used:
            dims = list(init.dims) if init.dims else [1]
            print(f"    ❌ '{init.name}' shape={dims} → {int(np.prod(dims))} wasted params")

In [ ]:
if worst_tasks:
    cleaned = copy.deepcopy(network)
    
    removed = []
    for init in list(cleaned.graph.initializer):
        if init.name not in used:
            cleaned.graph.initializer.remove(init)
            removed.append(init.name)
    
    onnx.checker.check_model(cleaned)
    score_after, mem_after, params_after = score_task(t, cleaned)
    
    !rm *.json 2>/dev/null
    !rm *.onnx 2>/dev/null
    
    print(f"\nTask {t:03d} AFTER cleanup:")
    print(f"  Parameters: {params_before} → {params_after}  (saved {params_before - params_after})")
    print(f"  Memory:     {mem_before} → {mem_after} bytes")
    print(f"  Score:      {score_before:.4f} → {score_after:.4f}  (Δ{score_after - score_before:+.4f})")
    print(f"  Removed:    {len(removed)} initializers")
    print(f"\n✅ Same computation. Same outputs. Higher score.")

<p style="background-color:#ede7f6; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #673ab7;
          border-radius:4px">
🚀 <strong>Step 3: Apply to all 400 tasks</strong><br>
Same operation, every model. Collect the total savings.
</p>

In [ ]:
def cleanup_unused_initializers(model):
    """Remove initializers not referenced by any node. Returns (model, saved_params) or (None, 0)."""
    graph = model.graph
    
    used = set()
    for node in graph.node:
        used.update(inp for inp in node.input if inp)
    
    to_remove = [init for init in graph.initializer if init.name not in used]
    
    if not to_remove:
        return None, 0
    
    saved = 0
    for init in to_remove:
        graph.initializer.remove(init)
        params = int(np.prod(init.dims)) if init.dims else 1
        saved += params
    
    return model, saved


def cleanup_all_tasks(task_dir=SRC):
    """Clean all .onnx files in a directory."""
    
    task_paths = sorted(glob.glob(f"{task_dir}/task*.onnx"))
    total_saved = 0
    tasks_cleaned = 0
    
    for path in tqdm(task_paths):
        task_num = int(os.path.basename(path).split(".")[0].replace("task",""))
        model = onnx.load(path)
        model, saved = cleanup_unused_initializers(model)
        
        if model is not None:
            onnx.checker.check_model(model)
            onnx.save(model, path)
            total_saved += saved
            tasks_cleaned += 1
            print(f"  Task {task_num:03d}: removed {saved} params")
    
    print(f"\n{'='*50}")
    print(f"✂️  {tasks_cleaned} tasks cleaned")
    print(f"📉 {total_saved} total params removed")
    print(f"✅ Zero computation changes")

cleanup_all_tasks()

$$ $$

## 🔗 Duplicate Weights

<p style="background-color:#ffe6f7; 
          padding:15px; 
          color:#111;
          font-size:16px;
          border-width:3px; 
          border-color:#f5dce9; 
          border-style:solid;
          border-radius:6px">
🔗 <strong>Deduplicate Identical Initializers</strong><br><br>
Different nodes sometimes carry their own copy of the <em>exact same</em> constant tensor — same dtype, same shape, same bytes, different name. The scorer counts each copy separately:<br><br>
Two copies of a <code>[30]</code> index array = <strong>60 params</strong>. One shared copy = <strong>30 params</strong>. Half the cost, same computation.<br><br>
<strong>What this step does:</strong><br>
1️⃣ Groups initializers by <code>(dtype, shape, raw bytes)</code><br>
2️⃣ Picks one canonical name per group<br>
3️⃣ Rewires all node inputs to the canonical<br>
4️⃣ Prunes the orphaned duplicates<br><br>
No network behavior changes. Same outputs, fewer params, <strong>higher score</strong>. Free points. 🎯
</p>

### Utils

In [ ]:
def initializer_key(init):
    """Fingerprint: (dtype, shape, raw bytes) — exact content match."""
    arr = numpy_helper.to_array(init)
    return (arr.dtype.str, tuple(arr.shape), arr.tobytes())

def find_duplicate_initializers(model):
    """Find groups of initializers with identical content."""
    groups = defaultdict(list)
    for init in model.graph.initializer:
        groups[initializer_key(init)].append(init.name)
    return [names for names in groups.values() if len(names) > 1]


def deduplicate_initializers(model):
    """Merge identical initializers. Rewire nodes to one canonical name. Prune orphans."""
    groups = defaultdict(list)
    for init in model.graph.initializer:
        groups[initializer_key(init)].append(init.name)
    
    replace = {}
    for names in groups.values():
        if len(names) <= 1:
            continue
        canonical = sorted(names, key=lambda s: (len(s), s))[0]
        for n in names:
            if n != canonical:
                replace[n] = canonical
    
    if not replace:
        return model, 0
    
    # Rewire node inputs
    for node in model.graph.node:
        for i, name in enumerate(node.input):
            if name in replace:
                node.input[i] = replace[name]
    
    # Prune orphaned duplicates
    used = {n for node in model.graph.node for n in node.input if n}
    kept = [i for i in model.graph.initializer if i.name in used]
    del model.graph.initializer[:]
    model.graph.initializer.extend(kept)
    
    return model, len(replace)

<p style="background-color:#fff3e0; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #ff9800;
          border-radius:4px">
🔍 <strong>Step 1: How many duplicates are hiding?</strong><br>
Scan all 400 models. Group initializers by exact content. Flag any group with more than one name.
</p>

In [ ]:
# Scan all 400 models
tasks_with_dups = []
for t in tqdm(range(1, 401)):
    model = onnx.load(f"{SRC}/task{t:03d}.onnx")
    dups = find_duplicate_initializers(model)
    if dups:
        total_wasted = sum(
            (len(group) - 1) * max(int(np.prod(
                numpy_helper.to_array(
                    next(i for i in model.graph.initializer if i.name == group[0])
                ).shape)), 1)
            for group in dups
        )
        tasks_with_dups.append((t, len(dups), total_wasted, dups))

tasks_with_dups.sort(key=lambda x: -x[2])

print(f"📊 {len(tasks_with_dups)} tasks have duplicate initializers\n")
print(f"{'Task':>6} {'Groups':>7} {'Wasted':>8}   Duplicate names (first group)")
print(f"{'─'*6} {'─'*7} {'─'*8}   {'─'*40}")
for task_num, n_groups, wasted, dups in tasks_with_dups[:15]:
    first = dups[0]
    preview = " = ".join(f"'{n}'" for n in first[:3])
    if len(first) > 3:
        preview += f" +{len(first)-3} more"
    print(f"  {task_num:03d}    {n_groups:>5}   {wasted:>6}   {preview}")

<p style="background-color:#e8f5e9; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #4caf50;
          border-radius:4px">
🔬 <strong>Step 2: Merge one, prove it works</strong><br>
Pick the task with the most duplicates. Rewire node inputs to one shared initializer. Verify score improves and outputs are identical.
</p>

In [ ]:
if tasks_with_dups:
    t = tasks_with_dups[0][0]
    network = onnx.load(f"{SRC}/task{t:03d}.onnx")
    
    score_before, mem_before, params_before = score_task(t, network)
    print(f"Task {t:03d} BEFORE dedup:")
    print(f"  Parameters: {params_before}")
    print(f"  Score:      {score_before:.4f}")
    
    # Show what we found
    name_to_init = {i.name: i for i in network.graph.initializer}
    dups = find_duplicate_initializers(network)
    print(f"  Duplicate groups: {len(dups)}")
    for group in dups[:5]:
        arr = numpy_helper.to_array(name_to_init[group[0]])
        saved = (len(group) - 1) * max(int(np.prod(arr.shape)), 1)
        print(f"    {len(group)} copies, shape={arr.shape} dtype={arr.dtype} → saves {saved} params")
    
    
    optimized = copy.deepcopy(network)
    optimized, n_merged = deduplicate_initializers(optimized)
    onnx.checker.check_model(optimized)
    
    score_after, mem_after, params_after = score_task(t, optimized)
    
    !rm *.json 2>/dev/null
    !rm *.onnx 2>/dev/null
    
    print(f"\nTask {t:03d} AFTER dedup:")
    print(f"  Parameters: {params_before} → {params_after}  (saved {params_before - params_after})")
    print(f"  Score:      {score_before:.4f} → {score_after:.4f}  (Δ{score_after - score_before:+.4f})")
    print(f"  Merged:     {n_merged} duplicate initializers")
    print(f"\n✅ Same computation. Same outputs. Higher score.")

<p style="background-color:#ede7f6; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #673ab7;
          border-radius:4px">
🚀 <strong>Step 3: Deduplicate all 400 tasks</strong><br>
Same merge across every model. Collect the total savings.
</p>

In [ ]:
"""Deduplicate identical initializers across all tasks."""

task_paths = sorted(glob.glob(f"{SRC}/task*.onnx"))
total_saved = 0
tasks_deduped = 0

for path in tqdm(task_paths):
    task_num = int(os.path.basename(path).split(".")[0].replace("task",""))
    model = onnx.load(path)
    
    dups = find_duplicate_initializers(model)
    if not dups:
        continue
    
    params_before = sum(max(int(np.prod(i.dims)), 1) for i in model.graph.initializer)
    model, n_merged = deduplicate_initializers(model)
    params_after = sum(max(int(np.prod(i.dims)), 1) for i in model.graph.initializer)
    
    saved = params_before - params_after
    if saved > 0:
        onnx.checker.check_model(model)
        onnx.save(model, path)
        total_saved += saved
        tasks_deduped += 1
        print(f"  Task {task_num:03d}: {params_before:>6d} → {params_after:>6d}  saved {saved:>4d} params")

print(f"\n{'='*50}")
print(f"🔗 {tasks_deduped} tasks deduplicated")
print(f"📉 {total_saved} total params removed")
print(f"✅ Zero computation changes")

$$ $$

## 📡 Tensor Broadcasting

<p style="background-color:#e6ffe6; 
          padding:15px; 
          color:#111;
          font-size:16px;
          border-width:3px; 
          border-color:#c8f0c8; 
          border-style:solid;
          border-radius:6px">
📡 <strong>Compress Uniform Tensors via Broadcasting</strong><br><br>
Some initializers store the <em>same value</em> in every element — an all-zeros mask of shape <code>[1,1,30,1]</code>, an all-ones weight of shape <code>[1,1,1,30]</code>. The scorer counts every element:<br><br>
A <code>[1,1,30,1]</code> tensor of all zeros = <strong>30 params</strong>. A scalar <code>0.0</code> = <strong>1 param</strong>. ONNX ops like <code>Add</code>, <code>Mul</code>, <code>Where</code>, <code>Greater</code> broadcast scalars automatically — same result, 29 fewer params.<br><br>
<strong>What this step does:</strong><br>
1️⃣ Finds initializers where <em>every element is identical</em><br>
2️⃣ Checks that <strong>all</strong> consuming ops support scalar broadcasting<br>
3️⃣ Replaces the tensor with a single scalar<br>
4️⃣ Skips any tensor feeding shape-dependent ops (<code>Conv</code>, <code>Pad</code>, <code>Reshape</code>, <code>Gather</code>...)<br><br>
No network behavior changes. Same outputs, fewer params, <strong>higher score</strong>. Free points. 🎯
</p>

### Constants

In [ ]:
SAFE_OPS = {'Greater', 'Less', 'Equal', 'Add', 'Sub', 'Mul', 'Div',
            'Where', 'Max', 'Min', 'And', 'Or', 'Not', 'Clip',
            'LessOrEqual', 'GreaterOrEqual', 'Sum'}


### Utils

In [ ]:
def find_compressible_initializers(model):
    """Find initializers where all elements are identical."""
    arrs = {}
    for init in model.graph.initializer:
        arrs[init.name] = numpy_helper.to_array(init)
    
    consumers = defaultdict(list)
    for node in model.graph.node:
        for i, name in enumerate(node.input):
            if name in arrs:
                consumers[name].append(node.op_type)
    
    candidates = []
    for name, arr in arrs.items():
        size = max(int(np.prod(arr.shape)), 1)
        if size <= 1:
            continue
        flat = arr.ravel()
        if not np.all(flat == flat[0]):
            continue
        consumer_ops = set(consumers.get(name, []))
        is_safe = consumer_ops <= SAFE_OPS
        candidates.append({
            "name": name, "shape": arr.shape, "dtype": arr.dtype,
            "value": flat[0].item(), "saveable": size - 1,
            "consumers": consumer_ops, "safe": is_safe,
        })
    return candidates


def compress_uniform_initializers(model):
    """Replace uniform-value initializers with scalars when all consumers broadcast safely."""
    arrs = {}
    for init in model.graph.initializer:
        arrs[init.name] = numpy_helper.to_array(init)
    
    consumers = defaultdict(list)
    for node in model.graph.node:
        for i, name in enumerate(node.input):
            if name in arrs:
                consumers[name].append(node.op_type)
    
    total_saved = 0
    for init in model.graph.initializer:
        arr = arrs[init.name]
        size = max(int(np.prod(arr.shape)), 1)
        if size <= 1:
            continue
        flat = arr.ravel()
        if not np.all(flat == flat[0]):
            continue
        consumer_ops = set(consumers.get(init.name, []))
        if not consumer_ops <= SAFE_OPS:
            continue
        
        scalar = np.array(flat[0], dtype=arr.dtype)
        init.CopyFrom(numpy_helper.from_array(scalar, init.name))
        total_saved += size - 1
    
    return model, total_saved

<p style="background-color:#fff3e0; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #ff9800;
          border-radius:4px">
🔍 <strong>Step 1: How many uniform tensors are out there?</strong><br>
Scan all 400 models. Find initializers where every element is the same value. Separate safe (broadcast-friendly ops) from unsafe (shape-dependent ops).
</p>

In [ ]:
safe_total, unsafe_total = 0, 0
safe_tasks = []

for t in tqdm(range(1, 401)):
    model = onnx.load(f"{SRC}/task{t:03d}.onnx")
    candidates = find_compressible_initializers(model)
    
    task_safe = sum(c["saveable"] for c in candidates if c["safe"])
    task_unsafe = sum(c["saveable"] for c in candidates if not c["safe"])
    safe_total += task_safe
    unsafe_total += task_unsafe
    
    if task_safe > 0:
        safe_tasks.append((t, task_safe, [c for c in candidates if c["safe"]]))

safe_tasks.sort(key=lambda x: -x[1])

print(f"📊 Uniform tensors across 400 models:\n")
print(f"  ✅ Safe to compress (broadcast-friendly ops): {safe_total:>6d} params")
print(f"  ⚠️  Unsafe (shape-dependent ops):             {unsafe_total:>6d} params")
print(f"\n  {len(safe_tasks)} tasks have safe compressible tensors\n")
print(f"{'Task':>6} {'Saveable':>9}   Example tensor")
print(f"{'─'*6} {'─'*9}   {'─'*45}")
for task_num, saved, cands in safe_tasks[:15]:
    c = cands[0]
    print(f"  {task_num:03d}    {saved:>6}   '{c['name']}' {c['shape']} val={c['value']} → scalar")

<p style="background-color:#e8f5e9; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #4caf50;
          border-radius:4px">
🔬 <strong>Step 2: Compress one, prove it works</strong><br>
Pick the task with the most compressible params. Replace uniform tensors with scalars. Verify the score improves and outputs don't change.
</p>

In [ ]:
# Pick the worst offender
if safe_tasks:
    t = safe_tasks[0][0]
    network = onnx.load(f"{SRC}/task{t:03d}.onnx")
    
    score_before, mem_before, params_before = score_task(t, network)
    print(f"Task {t:03d} BEFORE compression:")
    print(f"  Parameters: {params_before}")
    print(f"  Score:      {score_before:.4f}")
    
    cands = safe_tasks[0][2]
    print(f"  Compressible tensors: {len(cands)}")
    for c in cands[:5]:
        ops = ", ".join(c["consumers"])
        print(f"    '{c['name']}' {c['shape']} val={c['value']} → saves {c['saveable']} params  [{ops}]")
    if len(cands) > 5:
        print(f"    ... +{len(cands)-5} more")

    try:
        optimized = copy.deepcopy(network)
        optimized, saved = compress_uniform_initializers(optimized)
        onnx.checker.check_model(optimized)
        
        score_after, mem_after, params_after = score_task(t, optimized)
        print(f"\nTask {t:03d} AFTER compression:")
        print(f"  Parameters: {params_before} → {params_after}  (saved {params_before - params_after})")
        print(f"  Score:      {score_before:.4f} → {score_after:.4f}  (Δ{score_after - score_before:+.4f})")
        print(f"\n✅ Same computation. Same outputs. Higher score.")
    
    except Exception as e:
        print(f"  Task {task_num:03d}: Error Compressing Network ({type(e).__name__})")
        
    !rm *.json 2>/dev/null
    !rm *.onnx 2>/dev/null


<p style="background-color:#ede7f6; 
          padding:12px; 
          color:#111;
          font-size:14px;
          border-left:4px solid #673ab7;
          border-radius:4px">
🚀 <strong>Step 3: Compress all 400 tasks</strong><br>
Same scalar replacement across every model. Only touch tensors with broadcast-safe consumers.
</p>

In [ ]:
"""Compress uniform initializers to scalars across all tasks."""

SKIP_TASKS = {158, 54}  # Known to fail on grader despite passing locally

task_paths = sorted(glob.glob(f"{SRC}/task*.onnx"))
total_saved = 0
tasks_compressed = 0

for path in tqdm(task_paths):
    task_num = int(os.path.basename(path).split(".")[0].replace("task",""))

    if task_num in SKIP_TASKS:
        print(f"  Task {task_num:03d}: SKIPPED (known grader incompatibility)")
        continue

    original = onnx.load(path)
    model = copy.deepcopy(original)
    
    model, saved = compress_uniform_initializers(model)
    
    if saved > 0:
        try:
            onnx.checker.check_model(model)
            # Extra safety: verify shape inference still works
            onnx.shape_inference.infer_shapes(model, strict_mode=True)
            onnx.save(model, path)
            total_saved += saved
            tasks_compressed += 1
            print(f"  Task {task_num:03d}: saved {saved:>5d} params")
        except Exception as e:
            # Revert — save original back
            onnx.save(original, path)
            print(f"  Task {task_num:03d}: REVERTED ({type(e).__name__})")

print(f"\n{'='*50}")
print(f"📡 {tasks_compressed} tasks compressed")
print(f"📉 {total_saved} total params removed")
print(f"✅ Zero computation changes")

$$ $$

# 📤 Submit

In [ ]:
import zipfile

source_folder = '/kaggle/working/submission' 
output_filename = '/kaggle/working/submission.zip'

with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(source_folder):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, os.path.join(source_folder, '..')))

In [ ]:
ls